Implementazione Adam

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8,
)

for epoch in range(num_epochs):
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        loss = criterion(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()

Implementazione Batch Normalization

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(512, 256),
    nn.BatchNorm1d(256), # per dati tabulari / FC layers
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU()
)

Implementazione Dropout

In [ ]:
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.drop1 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(512, 256)
        self.drop2 = nn.Dropout(p=0.5)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.drop1(x) # attivo solo in training mode
        x = torch.relu(self.fc2(x))
        x = self.drop2(x)
        return self.fc3(x)

model = Classifier()
model.train() # abilita dropout
model.eval() # disabilita dropout

Transfer Learning con ResNet-50 in PyTorch: Feature Extraction e Fine-tuning

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T

NUM_CLASSES = 10 # numero di classi nel dataset obiettivo

# Caricamento del backbone pre-addestrato
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Paradigma 1: Feature Extraction # Congelo tutti i parametri del backbone
for param in backbone.parameters():
    param.requires_grad = False

# Sostituisco il classificatore finale
in_features = backbone.fc.in_features # 2048 per ResNet-50
backbone.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

# Solo i parametri di backbone.fc verranno aggiornati
optimizer_fe = torch.optim.Adam(
    backbone.fc.parameters(), lr=1e-3
)

# Paradigma 2: Fine-tuning (sblocco progressivo) Ricarico il backbone con tutti i pesi sbloccati
backbone_ft = models.resnet50(
    weights=models.ResNet50_Weights.IMAGENET1K_V1
)
backbone_ft.fc = nn.Linear(
    backbone_ft.fc.in_features, NUM_CLASSES
)

# Layer groups: layer4 e fc con lr piu' alto; layer1-3 con lr ridotto
optimizer_ft = torch.optim.SGD([
    {'params': backbone_ft.layer1.parameters(), 'lr': 1e-5},
    {'params': backbone_ft.layer2.parameters(), 'lr': 1e-5},
    {'params': backbone_ft.layer3.parameters(), 'lr': 1e-4},
    {'params': backbone_ft.layer4.parameters(), 'lr': 1e-4},
    {'params': backbone_ft.fc.parameters(), 'lr': 1e-3},
], momentum=0.9, weight_decay=1e-4)

# Pipeline di data augmentation per il training
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomRotation(degrees=15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], # media ImageNet
                std=[0.229, 0.224, 0.225]), # std ImageNet
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Classificatore LSTM bidirezionale con gestione del padding

In [ ]:
import torch
import torch.nn as nn

class BiLSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.5):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )

        # * 2 perche' bidirezionale
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        out, (hn, _) = self.lstm(packed)

        # Concatenazione stati finali: forward (hn[-2]) + backward (hn[-1])
        last = torch.cat([hn[-2], hn[-1]], dim=-1)

        return self.classifier(last)